In [1]:
!pip install opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 13.9 MB/s  0:00:03m0:00:0100:01


In [2]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'Phase 3' else Path.cwd().resolve()
DATASET_ROOT = PROJECT_ROOT / 'dataset'
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import numpy as np
import cv2
import os
import pandas as pd
from tqdm import tqdm
from PIL import Image
import random

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()


In [4]:
import torch
import torch.nn as nn
import torchvision.models as models
from pathlib import Path

DATASET_ROOT = PROJECT_ROOT / 'dataset'

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = PROJECT_ROOT / 'Phase1' / 'cifake_resnet18_final.pt'

model = models.resnet18(weights=None)

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 2)
)

state_dict = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state_dict)

model.to(device)
model.eval()

print("Model loaded successfully.")

NameError: name 'PROJECT_ROOT' is not defined

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [ ]:
def get_prediction(model, image_tensor):
    image_tensor = image_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(image_tensor)
        probs = F.softmax(output, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred].item()
    return pred, confidence

In [ ]:
def apply_blur(image_np, radius):
    k = int(max(1, radius * 5))
    if k % 2 == 0:
        k += 1
    blurred = cv2.GaussianBlur(image_np, (k, k), 0)
    return blurred

In [ ]:
def apply_frequency_mask(image_np, strength):
    img_float = image_np.astype(np.float32) / 255.0
    dft = np.fft.fft2(img_float, axes=(0,1))
    dft_shift = np.fft.fftshift(dft, axes=(0,1))

    h, w, _ = img_float.shape
    mask = np.ones((h, w, 3), np.float32)

    center_h, center_w = h // 2, w // 2
    radius = int(min(h, w) * strength)

    mask[center_h-radius:center_h+radius,
         center_w-radius:center_w+radius] *= 0.5

    dft_shift = dft_shift * mask
    f_ishift = np.fft.ifftshift(dft_shift, axes=(0,1))
    img_back = np.fft.ifft2(f_ishift, axes=(0,1))
    img_back = np.abs(img_back)

    img_back = np.clip(img_back, 0, 1)
    img_back = (img_back * 255).astype(np.uint8)

    return img_back

In [ ]:
def apply_noise(image_np, sigma):
    noise = np.random.normal(0, sigma * 255, image_np.shape)
    noisy = image_np + noise
    noisy = np.clip(noisy, 0, 255).astype(np.uint8)
    return noisy

In [ ]:
def apply_brightness(image_np, factor):
    bright = image_np.astype(np.float32) * factor
    bright = np.clip(bright, 0, 255).astype(np.uint8)
    return bright

In [ ]:
# Strategy A
blur_levels = [0.3, 0.5, 0.8, 1.0]
freq_levels = [0.05, 0.1, 0.15]

# Strategy B
noise_levels = [0.01, 0.02, 0.03, 0.05]
brightness_levels = [1.02, 1.05, 1.08]

In [ ]:
TEST_FOLDER = str(DATASET_ROOT / 'test')

results = []

MAX_SAMPLES = 100
counter = 0

for class_name in os.listdir(TEST_FOLDER):

    class_path = os.path.join(TEST_FOLDER, class_name)

    if not os.path.isdir(class_path):
        continue

    for img_name in tqdm(os.listdir(class_path)):

        if counter >= MAX_SAMPLES:
            break

        if not img_name.lower().endswith((".jpg", ".png", ".jpeg")):
            continue

        img_path = os.path.join(class_path, img_name)

        image = Image.open(img_path).convert("RGB")
        image_np = np.array(image)

        # ORIGINAL
        image_tensor = transform(image)
        original_pred, original_conf = get_prediction(model, image_tensor)

        results.append({
            "sample_id": img_name,
            "true_label": class_name,
            "strategy": "original",
            "stage": "original",
            "parameter": 0,
            "confidence": original_conf,
            "flipped": False
        })

        # =========================
        # STRATEGY A
        # =========================

        for blur in blur_levels:
            blurred = apply_blur(image_np, blur)
            tensor = transform(Image.fromarray(blurred))
            pred, conf = get_prediction(model, tensor)

            results.append({
                "sample_id": img_name,
                "true_label": class_name,
                "strategy": "A_blur",
                "stage": f"blur_{blur}",
                "parameter": blur,
                "confidence": conf,
                "flipped": pred != original_pred
            })

        for freq in freq_levels:
            hybrid = apply_blur(image_np, 0.5)
            hybrid = apply_frequency_mask(hybrid, freq)

            tensor = transform(Image.fromarray(hybrid))
            pred, conf = get_prediction(model, tensor)

            results.append({
                "sample_id": img_name,
                "true_label": class_name,
                "strategy": "A_blur_freq",
                "stage": f"blur0.5_freq{freq}",
                "parameter": freq,
                "confidence": conf,
                "flipped": pred != original_pred
            })

        # =========================
        # STRATEGY B
        # =========================

        for sigma in noise_levels:
            noisy = apply_noise(image_np, sigma)
            tensor = transform(Image.fromarray(noisy))
            pred, conf = get_prediction(model, tensor)

            results.append({
                "sample_id": img_name,
                "true_label": class_name,
                "strategy": "B_noise",
                "stage": f"noise_{sigma}",
                "parameter": sigma,
                "confidence": conf,
                "flipped": pred != original_pred
            })

        for bright in brightness_levels:
            hybrid = apply_noise(image_np, 0.02)
            hybrid = apply_brightness(hybrid, bright)

            tensor = transform(Image.fromarray(hybrid))
            pred, conf = get_prediction(model, tensor)

            results.append({
                "sample_id": img_name,
                "true_label": class_name,
                "strategy": "B_noise_bright",
                "stage": f"noise0.02_bright{bright}",
                "parameter": bright,
                "confidence": conf,
                "flipped": pred != original_pred
            })

        counter += 1

    if counter >= MAX_SAMPLES:
        break

print(f"Processed {counter} images.")

  1%|          | 100/10000 [02:44<4:31:35,  1.65s/it]

Processed 100 images.


In [ ]:
os.makedirs("outputs", exist_ok=True)

df = pd.DataFrame(results)
df.to_csv("outputs/strategy_results.csv", index=False)

print("Phase 3 – M1 execution completed.")
print("Saved to outputs/strategy_results.csv")

Phase 3 – M1 execution completed.
Saved to outputs/strategy_results.csv
